In [1]:
import sys
sys.path.append('../../../../')

In [2]:
import numpy as np

from CADETProcess.processModel import ComponentSystem
component_system = ComponentSystem(2)

from CADETProcess.processModel import MassActionLaw
reaction_system = MassActionLaw(component_system)
reaction_system.add_reaction(
    indices=[0, 1],
    coefficients=[-1, 1],
    k_fwd=0.1,
    k_bwd=0,
)
reaction_system.add_reaction(
    indices=[1, 0],
    coefficients=[-1, 1],
    k_fwd=0.2,
    k_bwd=0,
)

from CADETProcess.processModel import Inlet, LumpedRateModelWithPores, Outlet
inlet = Inlet(component_system, name='inlet')
column = LumpedRateModelWithPores(component_system, 'column')
column.bulk_reaction_model = reaction_system
outlet = Outlet(component_system, 'outlet')

from CADETProcess.processModel import FlowSheet, Process

flow_sheet = FlowSheet(component_system)
flow_sheet.add_unit(inlet)
flow_sheet.add_unit(column)
flow_sheet.add_unit(outlet)

flow_sheet.add_connection(inlet, column)
flow_sheet.add_connection(column, outlet)

def setup_process():
    process = Process(flow_sheet, 'Demo Indices')
    process.cycle_time = 10

    return process

from CADETProcess.optimization import OptimizationProblem

def setup_optimization_problem():
    optimization_problem = OptimizationProblem('Demo Indices', use_diskcache=False)
    optimization_problem.add_evaluation_object(process)

    return optimization_problem

[INFO 08-12 16:22:48] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.52


In [3]:
process = setup_process()
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'film_diffusion_all', evaluation_objects=process, parameter_path='flow_sheet.column.film_diffusion'
)
optimization_problem.set_variables([1])
print(process.flow_sheet.column.film_diffusion)
assert np.allclose(process.flow_sheet.column.film_diffusion, [1, 1])

[1.0, 1.0]


In [4]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'film_diffusion_0', evaluation_objects=process, parameter_path='flow_sheet.column.film_diffusion', indices=0
)
optimization_problem.set_variables([2])
print(process.flow_sheet.column.film_diffusion)
assert np.allclose(process.flow_sheet.column.film_diffusion, [2, 1])

[2.0, 1.0]


In [5]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'flow_rate_fill', evaluation_objects=process, parameter_path='flow_sheet.inlet.flow_rate'
)
optimization_problem.set_variables([1])
print(process.flow_sheet.inlet.flow_rate)
assert np.allclose(process.flow_sheet.inlet.flow_rate, [1, 0, 0, 0])

[1. 0. 0. 0.]


In [6]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'flow_rate_single', evaluation_objects=process, parameter_path='flow_sheet.inlet.flow_rate', indices=1
)
optimization_problem.set_variables([2])
print(process.flow_sheet.inlet.flow_rate)
assert np.allclose(process.flow_sheet.inlet.flow_rate, [1, 2, 0, 0])

[1. 2. 0. 0.]


In [7]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'exponents', evaluation_objects=process,
    parameter_path='flow_sheet.column.bulk_reaction_model.exponents_fwd', indices=np.s_[:, :]
)
optimization_problem.set_variables([1])
print(process.flow_sheet.column.bulk_reaction_model.exponents_fwd)
assert np.allclose(process.flow_sheet.column.bulk_reaction_model.exponents_fwd, [[1, 1], [1, 1]])

[[1. 1.]
 [1. 1.]]


In [8]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'exponents_single', evaluation_objects=process,
    parameter_path='flow_sheet.column.bulk_reaction_model.exponents_fwd', indices=(0, 0)
)
optimization_problem.set_variables([2])
print(process.flow_sheet.column.bulk_reaction_model.exponents_fwd)
assert np.allclose(process.flow_sheet.column.bulk_reaction_model.exponents_fwd, [[2, 1], [1, 1]])

[[2. 1.]
 [1. 1.]]


In [9]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'exponents_slice', evaluation_objects=process,
    parameter_path='flow_sheet.column.bulk_reaction_model.exponents_fwd', indices=np.s_[0, :]
)
optimization_problem.set_variables([3])
print(process.flow_sheet.column.bulk_reaction_model.exponents_fwd)
assert np.allclose(process.flow_sheet.column.bulk_reaction_model.exponents_fwd, [[3, 3], [1, 1]])

[[3. 3.]
 [1. 1.]]


In [10]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'concentration_constant_all', evaluation_objects=process, parameter_path='flow_sheet.inlet.c'
)
optimization_problem.set_variables([1])
print(process.flow_sheet.inlet.c)
assert np.allclose(process.flow_sheet.inlet.c, [[1, 0, 0, 0], [1, 0, 0, 0]])

[[1. 0. 0. 0.]
 [1. 0. 0. 0.]]


In [11]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'concentration_fill_values_single', evaluation_objects=process, parameter_path='flow_sheet.inlet.c', indices=0
)
optimization_problem.set_variables([2])
print(process.flow_sheet.inlet.c)
assert np.allclose(process.flow_sheet.inlet.c, [[2, 0, 0, 0], [1, 0, 0, 0]])

[[2. 0. 0. 0.]
 [1. 0. 0. 0.]]


In [12]:
optimization_problem = setup_optimization_problem()

optimization_problem.add_variable(
    'concentration_single_entry', evaluation_objects=process, parameter_path='flow_sheet.inlet.c', indices=(0, 1)
)
optimization_problem.set_variables([3])
print(process.flow_sheet.inlet.c)
assert np.allclose(process.flow_sheet.inlet.c, [[2, 3, 0, 0], [1, 0, 0, 0]])

[[2. 3. 0. 0.]
 [1. 0. 0. 0.]]


In [13]:
process = setup_process()
optimization_problem = setup_optimization_problem()

evt = process.add_event(
    'c_poly', 'flow_sheet.inlet.c', [[0, 1], 0], time=1
)
print(inlet.c)
assert np.allclose(inlet.c, [[0, 1, 0, 0], [0, 0, 0, 0]])

[[0. 1. 0. 0.]
 [0. 0. 0. 0.]]


In [14]:
optimization_problem.add_variable(
    'c_poly_linear', evaluation_objects=process, parameter_path='c_poly.state',
    pre_processing=lambda w: [[0, w], 0]
)
optimization_problem.set_variables([2])
print(evt.state)
assert evt.state == [[0, 2.0], 0]

[[0, 2.0], 0]
